# Part 1: Build and save the RAG vector store

**This notebook** reads **`RAG_docs/knowledge/`** (PDFs and optional JSON) and writes **`RAG_docs/vector_store/`** (FAISS + `rag_manifest.json`) plus **`rag_parents.json`**. Run **`RAG_part2_agent_actions.ipynb`** after a successful build.

## Index build (what runs before save)

- **Section merge**: PDF pages merged into heading-based sections.
- **Semantic parents**: paragraph-embedding similarity splits each section into parent chunks.
- **Retrieval title**: one title per parent (default: local HF summarizer; or extractive / LLM via `TITLE_MODE` in the code cell).
- **Child chunks + FAISS**: small chunks for search; each indexed vector is **`retrieval_title` + child body** (batched embedding via helpers where applicable).
- **Sidecar**: `rag_parents.json` stores full parent text and metadata so Part 2 can expand child hits to full sections.

Requires **`pypdf`** for PDFs. **Shared helpers:** `utils/rag_index_build.py`, `utils/rag_utils.py`. Re-run when KB files or chunking/embed settings change.

In [ ]:
import sys, warnings
if sys.version_info >= (3, 14):
    warnings.filterwarnings(
        "ignore",
        message="Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.",
        category=UserWarning,
    )

import json
from pathlib import Path
from typing import Any, Dict, List, Optional

from utils.env import load_project_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.embeddings import SentenceTransformerEmbeddings
from utils.rag_index_build import (
    CHILD_CHUNK_OVERLAP,
    CHILD_CHUNK_SIZE,
    TITLE_MODE_EXTRACTIVE,
    TITLE_MODE_LLM,
    TITLE_MODE_TRANSFORMER,
    build_documents_from_knowledge_base,
    expand_pdf_kb_items_with_sections,
    save_parent_store,
)
from utils.rag_utils import faiss_from_documents_batched, save_vector_store

load_project_dotenv()

try:
    import pypdf  # noqa: F401 — PyPDFLoader depends on this at runtime
except ImportError as e:
    raise ImportError(
        "PDF loading requires pypdf. Install with: pip install pypdf"
    ) from e

KNOWLEDGE_DIR = Path("RAG_docs/knowledge")
VECTOR_STORE_DIR = Path("RAG_docs/vector_store")
EMBED_MODEL = "all-MiniLM-L6-v2"
# sentence-transformers encode batch — set 8 or 4 if PyTorch reports CPU OOM
EMBED_ENCODE_BATCH = 16
# "transformer" = local HF summarizer (default) | "extractive" | "llm"
TITLE_MODE = TITLE_MODE_TRANSFORMER
SUMMARIZATION_MODEL = None  # or e.g. "sshleifer/distilbart-cnn-12-6"; env RAG_SUMMARY_MODEL overrides
PRINT_TITLES = True
MAX_LLM_PARENTS = 120  # only when TITLE_MODE == TITLE_MODE_LLM


def _normalize_kb_record(
    item: Dict[str, Any],
    source_file: str,
    doc_type: str,
    page: Optional[int] = None,
) -> Dict[str, Any]:
    out = dict(item)
    out["source_file"] = source_file
    out["doc_type"] = doc_type
    if page is not None:
        out["page"] = page
    return out


def load_pdf_file(file_path: Path) -> List[Dict[str, Any]]:
    """One dict per PDF page with title, text, source_file, doc_type, page."""
    loader = PyPDFLoader(str(file_path))
    docs = loader.load()
    rows: List[Dict[str, Any]] = []
    for i, doc in enumerate(docs):
        meta_page = doc.metadata.get("page")
        page_1based = int(meta_page) + 1 if meta_page is not None else i + 1
        rows.append(
            _normalize_kb_record(
                {
                    "title": f"{file_path.name} (page {page_1based})",
                    "text": doc.page_content or "",
                },
                file_path.name,
                "pdf",
                page=page_1based,
            )
        )
    return rows


def load_knowledge_base(knowledge_dir: Path, verbose: bool = True) -> List[Dict[str, Any]]:
    """Load all *.json and *.pdf into flat records with title, text, source_file, doc_type, page (PDF)."""
    kb: List[Dict[str, Any]] = []
    knowledge_json = sorted(knowledge_dir.glob("*.json"))
    knowledge_pdf = sorted(knowledge_dir.glob("*.pdf"))
    if not knowledge_json and not knowledge_pdf:
        raise FileNotFoundError(f"No JSON or PDF files found in {knowledge_dir}")

    if verbose:
        print(f"Loading knowledge base from {knowledge_dir}:")

    for knowledge_file in knowledge_json:
        filename = knowledge_file.name
        try:
            with open(knowledge_file, "r", encoding="utf-8") as f:
                data = json.load(f)
            if isinstance(data, list):
                n = 0
                for row in data:
                    if isinstance(row, dict) and "title" in row and "text" in row:
                        kb.append(_normalize_kb_record(row, filename, "json"))
                        n += 1
                if verbose:
                    print(f"  Loaded {n} document(s) from {filename}")
            elif isinstance(data, dict) and "title" in data and "text" in data:
                kb.append(_normalize_kb_record(data, filename, "json"))
                if verbose:
                    print(f"  Loaded 1 document from {filename}")
            elif verbose:
                print(f"  Skipped {filename} (unknown format)")
        except Exception as e:
            if verbose:
                print(f"  Error loading {filename}: {e}")

    for pdf_file in knowledge_pdf:
        try:
            docs = load_pdf_file(pdf_file)
            kb.extend(docs)
            if verbose:
                print(f"  Loaded {len(docs)} document(s) from {pdf_file.name} (PDF)")
        except Exception as e:
            if verbose:
                print(f"  Error loading {pdf_file.name}: {e}")

    if verbose:
        print(f"Total: {len(kb)} knowledge documents")
    return kb


# --- PDF pages → sections → semantic parents → child chunks → FAISS -----------------
knowledge_base = load_knowledge_base(KNOWLEDGE_DIR)
knowledge_expanded = expand_pdf_kb_items_with_sections(
    knowledge_base, KNOWLEDGE_DIR, verbose=True
)
embeddings = SentenceTransformerEmbeddings(
    model_name=EMBED_MODEL,
    encode_kwargs={"batch_size": EMBED_ENCODE_BATCH},
)
rag_documents, parent_store = build_documents_from_knowledge_base(
    knowledge_expanded,
    embeddings,
    title_mode=TITLE_MODE,
    summarization_model=SUMMARIZATION_MODEL,
    print_titles=PRINT_TITLES,
    max_llm_parents=MAX_LLM_PARENTS,
    verbose=True,
)

print(
    f"Embedding model={EMBED_MODEL}, child chunk={CHILD_CHUNK_SIZE}/{CHILD_CHUNK_OVERLAP}, "
    f"{len(rag_documents)} FAISS vectors, {len(parent_store)} parents"
)

vector_store = faiss_from_documents_batched(rag_documents, embeddings)

save_vector_store(
    vector_store,
    VECTOR_STORE_DIR,
    knowledge_expanded,
    embed_model=EMBED_MODEL,
    chunk_size=CHILD_CHUNK_SIZE,
    chunk_overlap=CHILD_CHUNK_OVERLAP,
    n_chunks=len(rag_documents),
    extra_manifest={
        "indexing": "parent_child_semantic",
        "n_parents": len(parent_store),
        "title_mode": TITLE_MODE,
        "max_llm_parents": MAX_LLM_PARENTS,
    },
)
ps_path = save_parent_store(VECTOR_STORE_DIR, parent_store)
print(f"Saved vector store to {VECTOR_STORE_DIR}")
print(f"Saved parent store to {ps_path}")

In [ ]:
# Optional: smoke-test retrieval (edit query)
# L2 distance → heuristic score; metadata "retrieval_title" reflects TITLE_MODE (transformer / extractive / llm).
q = "DDoS mitigation NIST zero trust access control"
pairs = vector_store.similarity_search_with_score(q, k=5)
print(f"Query: {q!r} -> {len(pairs)} hits")
for i, (doc, dist) in enumerate(pairs, 1):
    rt = (doc.metadata.get("retrieval_title") or doc.metadata.get("title") or "")[:88]
    src = doc.metadata.get("source_file") or "?"
    pid = doc.metadata.get("parent_id") or "-"
    score = 1.0 / (1.0 + float(dist))
    print(f"  [{i}] {rt}... | src={src} parent={pid[:12]}... | score={score:.4f}")